In [0]:
%run ../../utils/utils

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import pyspark.sql.functions as F

##  Ler as 3 tabelas Gold consolidadas

In [0]:
pdf_regras = ler_delta("gold", "gold_dq_regras_consolidado", STORAGE_OPTIONS).toPandas()
pdf_saude_hora = ler_delta("gold", "gold_dq_saude_consolidado", STORAGE_OPTIONS).toPandas()
pdf_saude_pct = ler_delta("gold", "gold_saude_percentual_consolidado", STORAGE_OPTIONS).toPandas()
 
print(f"Linhas em gold_dq_regras_consolidado: {len(pdf_regras)}")
print(f"Linhas em gold_dq_saude_consolidado: {len(pdf_saude_hora)}")
print(f"Linhas em gold_saude_percentual_consolidado: {len(pdf_saude_pct)}")

##  Ler também a tabela Gold de Volumetria (Bronze x Silver x Quarentena)

In [0]:
pdf_volumetria = ler_delta("gold", "gold_volumetria_tabelas", STORAGE_OPTIONS).toPandas()
print(f"Linhas em gold_volumetria_tabelas: {len(pdf_volumetria)}")

##  Resumo geral — saúde de todas as tabelas lado a lado

In [0]:
resumo = pdf_saude_pct[[
    "tabela", "total_bronze_distintos", "total_quarentena",
    "percentual_saude", "percentual_invalidado"
]].sort_values("percentual_invalidado", ascending=False)
 
print("===== RESUMO GERAL DE SAÚDE POR TABELA =====")
display(resumo)
 
pior_tabela = resumo.iloc[0]
melhor_tabela = resumo.iloc[-1]
print(f"\nPior saúde: {pior_tabela['tabela']} ({pior_tabela['percentual_invalidado']}% invalidado)")
print(f"Melhor saúde: {melhor_tabela['tabela']} ({melhor_tabela['percentual_invalidado']}% invalidado)")

##  Ranking de saúde por tabela (gráfico)

In [0]:
resumo_ordenado = resumo.sort_values("percentual_saude", ascending=True)
 
fig, ax = plt.subplots(figsize=(10, max(4, len(resumo_ordenado) * 0.6)))
cores = ["#C44E52" if v < 80 else "#DD8452" if v < 95 else "#55A868" for v in resumo_ordenado["percentual_saude"]]
ax.barh(resumo_ordenado["tabela"], resumo_ordenado["percentual_saude"], color=cores)
ax.set_xlabel("% Saúde (passou para a Silver)")
ax.set_title("Ranking de Saúde por Tabela")
ax.set_xlim(0, 100)
for i, v in enumerate(resumo_ordenado["percentual_saude"]):
    ax.text(v + 1, i, f"{v}%", va="center", fontweight="bold")
plt.tight_layout()
plt.show()

##  Top 10 regras com mais falhas (somado em todas as tabelas e datas)

In [0]:
if not pdf_regras.empty:
    top_regras = (
        pdf_regras.groupby(["tabela", "regra", "severidade"])["total_falhas"]
        .sum()
        .reset_index()
        .sort_values("total_falhas", ascending=False)
        .head(10)
    )
 
    print("===== TOP 10 REGRAS COM MAIS FALHAS (todas as tabelas) =====")
    display(top_regras)
 
    fig, ax = plt.subplots(figsize=(10, 6))
    rotulos = top_regras["tabela"] + " | " + top_regras["regra"]
    cores_sev = ["#C44E52" if s == "Critica" else "#DD8452" for s in top_regras["severidade"]]
    ax.barh(rotulos[::-1], top_regras["total_falhas"][::-1], color=cores_sev[::-1])
    ax.set_xlabel("Total de falhas")
    ax.set_title("Top 10 Regras com Mais Falhas (todas as tabelas)")
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum log de falha encontrado em nenhuma tabela.")

##  Falhas por tabela, separadas por severidade (Crítica vs Aviso)

In [0]:
if not pdf_regras.empty:
    falhas_por_tabela_severidade = (
        pdf_regras.groupby(["tabela", "severidade"])["total_falhas"]
        .sum()
        .unstack(fill_value=0)
    )
 
    print("===== FALHAS POR TABELA E SEVERIDADE =====")
    display(falhas_por_tabela_severidade.reset_index())
 
    falhas_por_tabela_severidade.plot(
        kind="barh", stacked=True, figsize=(10, max(4, len(falhas_por_tabela_severidade) * 0.6)),
        color={"Critica": "#C44E52", "Aviso": "#DD8452"}
    )
    plt.xlabel("Total de falhas")
    plt.title("Falhas por Tabela (Crítica vs Aviso)")
    plt.tight_layout()
    plt.show()

##  Volume de registros limpos ao longo do tempo, por tabela

In [0]:
if not pdf_saude_hora.empty:
    pdf_saude_hora["data_hora"] = pd.to_datetime(pdf_saude_hora["data_hora"])
 
    fig, ax = plt.subplots(figsize=(12, 6))
    for tabela, grupo in pdf_saude_hora.groupby("tabela"):
        grupo_ordenado = grupo.sort_values("data_hora")
        ax.plot(grupo_ordenado["data_hora"], grupo_ordenado["qtd_limpos"], marker="o", label=tabela)
 
    ax.set_xlabel("Data/Hora")
    ax.set_ylabel("Registros limpos (Silver)")
    ax.set_title("Volume de Registros Limpos ao Longo do Tempo, por Tabela")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum dado de saúde por hora encontrado.")


##  Alertas — tabelas que precisam de atenção

In [0]:
LIMIAR_ATENCAO = 80.0  # % de saúde abaixo disso vira alerta
 
tabelas_criticas = resumo[resumo["percentual_saude"] < LIMIAR_ATENCAO]
 
if not tabelas_criticas.empty:
    print(f"⚠️  {len(tabelas_criticas)} tabela(s) abaixo de {LIMIAR_ATENCAO}% de saúde:\n")
    for _, row in tabelas_criticas.iterrows():
        print(f"  - {row['tabela']}: {row['percentual_saude']}% saudável "
              f"({row['total_quarentena']} de {row['total_bronze_distintos']} invalidados)")
else:
    print(f"✅ Todas as tabelas estão acima de {LIMIAR_ATENCAO}% de saúde.")

##  Funil de Dados — Bronze → Silver → Quarentena, por tabela

Mostra, lado a lado, quantos registros entraram (Bronze), quantos passaram na qualidade (Silver)
e quantos ficaram retidos por violarem alguma regra (Quarentena). É o gráfico mais direto para
mostrar ao time de negócio "quanto do dado bruto vira dado confiável".

In [0]:
if not pdf_volumetria.empty:
    funil = pdf_volumetria[[
        "nome_tabela_origem", "total_registros_bronze_distintos",
        "total_registros_silver", "total_registros_quarentena"
    ]].rename(columns={
        "nome_tabela_origem": "tabela",
        "total_registros_bronze_distintos": "Bronze (distintos)",
        "total_registros_silver": "Silver (aprovados)",
        "total_registros_quarentena": "Quarentena (reprovados)"
    })

    print("===== FUNIL DE DADOS POR TABELA =====")
    display(funil)

    fig, ax = plt.subplots(figsize=(11, max(4, len(funil) * 0.7)))
    funil.set_index("tabela").plot(kind="barh", ax=ax, color=["#4C72B0", "#55A868", "#C44E52"])
    ax.set_xlabel("Quantidade de registros")
    ax.set_title("Funil de Dados: Bronze → Silver → Quarentena")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()
else:
    print("gold_volumetria_tabelas vazia ou não encontrada — rode o notebook volumetria.py antes deste.")

##  Taxa de Aproveitamento — quanto da Bronze realmente virou dado confiável
Métrica-síntese para apresentação executiva: % da Bronze que efetivamente chegou limpa na Silver.

In [0]:
if not pdf_volumetria.empty:
    comparativo = pdf_volumetria.copy()
    comparativo["perc_aproveitamento"] = (
        comparativo["total_registros_silver"] / comparativo["total_registros_bronze_distintos"] * 100
    ).round(2)
    comparativo = comparativo[[
        "nome_tabela_origem", "total_registros_bronze_distintos", "total_registros_silver",
        "total_registros_quarentena", "perc_aproveitamento"
    ]].sort_values("perc_aproveitamento")

    print("===== TAXA DE APROVEITAMENTO (Silver / Bronze) POR TABELA =====")
    display(comparativo)

    fig, ax = plt.subplots(figsize=(10, max(4, len(comparativo) * 0.6)))
    cores = ["#C44E52" if v < 80 else "#DD8452" if v < 95 else "#55A868" for v in comparativo["perc_aproveitamento"]]
    ax.barh(comparativo["nome_tabela_origem"], comparativo["perc_aproveitamento"], color=cores)
    ax.set_xlim(0, 100)
    ax.set_xlabel("% de aproveitamento (Silver / Bronze)")
    ax.set_title("Taxa de Aproveitamento por Tabela")
    for i, v in enumerate(comparativo["perc_aproveitamento"]):
        ax.text(v + 1, i, f"{v}%", va="center", fontweight="bold")
    plt.tight_layout()
    plt.show()


##  Proporção Geral de Falhas por Severidade

Visão executiva: de tudo que falhou em todas as tabelas, qual fatia é crítica vs aviso.

In [0]:
if not pdf_regras.empty:
    dist_sev = pdf_regras.groupby("severidade")["total_falhas"].sum()
    mapa_cores = {"Critica": "#C44E52", "Aviso": "#DD8452"}
    cores_pizza = [mapa_cores.get(s, "#8172B2") for s in dist_sev.index]

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.pie(dist_sev, labels=dist_sev.index, autopct="%1.1f%%", colors=cores_pizza, startangle=90)
    ax.set_title("Distribuição de Falhas por Severidade (todas as tabelas)")
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum dado de falhas encontrado para calcular a distribuição por severidade.")

##  Evolução da Taxa de Falha ao Longo do Tempo, por tabela

Só aparece tendência de fato quando já existir mais de uma data de execução no histórico de logs.

In [0]:
if not pdf_regras.empty and "data_execucao" in pdf_regras.columns:
    evolucao = (
        pdf_regras.groupby(["data_execucao", "tabela"])
        .apply(lambda g: round((g["total_falhas"].sum() / g["total_processado"].sum()) * 100, 2))
        .reset_index(name="perc_falha_dia")
    )
    evolucao["data_execucao"] = pd.to_datetime(evolucao["data_execucao"])

    if evolucao["data_execucao"].nunique() > 1:
        fig, ax = plt.subplots(figsize=(12, 6))
        for tabela, grupo in evolucao.groupby("tabela"):
            grupo_ordenado = grupo.sort_values("data_execucao")
            ax.plot(grupo_ordenado["data_execucao"], grupo_ordenado["perc_falha_dia"], marker="o", label=tabela)
        ax.set_xlabel("Data de Execução")
        ax.set_ylabel("% de Falha")
        ax.set_title("Evolução da Taxa de Falha por Tabela")
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        plt.show()
    else:
        print("Apenas uma data de execução disponível até o momento — a tendência aparece "
              "conforme o pipeline rodar em dias diferentes.")
else:
    print("Sem dados de regras/execuções para calcular a evolução da taxa de falha.")

##  Resumo Executivo — Principais Insights para o Time de Negócio

Narrativa em texto, pronta para colar em slide, cobrindo os números que mais importam
para quem não vai olhar os gráficos em detalhe.

In [0]:
print("===== RESUMO EXECUTIVO =====\n")

if not pdf_volumetria.empty:
    total_bronze = int(pdf_volumetria["total_registros_bronze_distintos"].sum())
    total_silver = int(pdf_volumetria["total_registros_silver"].sum())
    total_quarentena = int(pdf_volumetria["total_registros_quarentena"].sum())
    perc_geral = round((total_silver / total_bronze) * 100, 2) if total_bronze else 0

    print(f"• Volume total processado (Bronze, distintos): {total_bronze:,} registros")
    print(f"• Registros validados e confiáveis (Silver): {total_silver:,} ({perc_geral}%)")
    print(f"• Registros que precisam de correção (Quarentena): {total_quarentena:,} "
          f"({round(100 - perc_geral, 2)}%)")

if not resumo.empty:
    print(f"\n• Tabela com melhor saúde: {melhor_tabela['tabela']} "
          f"({melhor_tabela['percentual_saude']}% saudável)")
    print(f"• Tabela que mais precisa de atenção: {pior_tabela['tabela']} "
          f"({pior_tabela['percentual_saude']}% saudável)")

if not pdf_regras.empty:
    falhas_criticas = pdf_regras[pdf_regras["severidade"] == "Critica"]
    if not falhas_criticas.empty:
        agrupado = falhas_criticas.groupby(["tabela", "regra"])["total_falhas"].sum()
        regra_top = agrupado.idxmax()
        valor_top = agrupado.max()
        print(f"\n• Falha mais recorrente: regra '{regra_top[1]}' na tabela '{regra_top[0]}' "
              f"({int(valor_top):,} ocorrências no histórico)")

if not tabelas_criticas.empty:
    print(f"\n⚠️  {len(tabelas_criticas)} tabela(s) abaixo de {LIMIAR_ATENCAO}% de saúde e que merecem "
          f"ação prioritária: {', '.join(tabelas_criticas['tabela'].tolist())}")
else:
    print(f"\n✅ Nenhuma tabela abaixo do limiar de {LIMIAR_ATENCAO}% — qualidade geral sob controle.")